# 03 — Human-in-the-loop via `HumanInTheLoopMiddleware`

`HumanInTheLoopMiddleware` is a **pre-built middleware** for one
specific pattern: pausing before a risky tool call so a human can
approve, edit, reject, or respond.

Under the hood it's just calling LangGraph's `interrupt()` for you —
see notebook **04** for the raw primitive underneath this.

**Requires a checkpointer** — human review isn't instantaneous, so the
graph state needs to be persisted while it waits.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


## 1. Define tools — one safe, one that should require approval

In [2]:
@tool
def read_data(table: str) -> str:
    """Read-only lookup — considered safe, no approval needed."""
    return f"[stub] {table} has 128 rows"


@tool
def execute_sql(query: str) -> str:
    """Execute a SQL statement against the warehouse. Potentially destructive."""
    return f"[stub] executed: {query}"


## 2. Wire up `HumanInTheLoopMiddleware`

`interrupt_on` maps each tool name to a policy:
- `True` → all decisions allowed (approve / edit / reject / respond)
- a dict with `allowed_decisions` → restrict which decisions are valid
- `False` → never interrupt, run automatically


In [ ]:
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[read_data, execute_sql],
    system_prompt="You are a data engineering assistant with warehouse access.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},
                "read_data": False,
            }
        ),
    ],
    checkpointer=checkpointer,
)


## 3. Trigger the interrupt

A `thread_id` is required — it's the key the checkpointer uses to
persist and later resume this exact conversation.


In [ ]:
config = {"configurable": {"thread_id": "demo-thread-1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete rows older than 30 days from the logs table."}]},
    config,
)

print("Interrupted:", bool(result.get("__interrupt__")))
result.get("__interrupt__")


## 4. Resume with a human decision

In a real app this is where you'd surface the pending action in a UI
and wait for a click. Here we simulate an approval.


In [ ]:
resumed = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,
)

for m in resumed["messages"]:
    print(f"[{m.type}] {m.content}")


### ...or reject it instead

Re-run from step 3 with a fresh `thread_id` if you want to try the
rejection path independently.


In [ ]:
config_reject = {"configurable": {"thread_id": "demo-thread-2"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "Delete rows older than 30 days from the logs table."}]},
    config_reject,
)

rejected = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Not approved — check with DBA first."}]}),
    config_reject,
)

print(rejected["messages"][-1].content)


## Recap

- `HumanInTheLoopMiddleware` is the fast path for the common
  "approve/edit/reject a tool call" pattern inside `create_agent`.
- It still requires LangGraph's checkpointer + `interrupt()` /
  `Command(resume=...)` mechanics underneath — the middleware just
  saves you from wiring that yourself.
- For anything outside "gate a tool call" (e.g. pause mid-plan for a
  human to pick a direction, or a fully custom multi-agent graph),
  you need the raw primitive — see notebook **04**.
